# Data Wrangling II — Academic Performance Dataset

We create a **synthetic Academic Performance dataset** of students and perform:
1. Missing value & inconsistency handling
2. Outlier detection and treatment
3. Data transformation (log transform to normalize skewed data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Create Academic Performance Dataset
n = 100
data = {
    'StudentID': range(1, n+1),
    'Age': np.random.randint(17, 25, n).astype(float),
    'Gender': np.random.choice(['Male', 'Female', 'M', 'F', None], n),  # inconsistency
    'Math_Score': np.random.randint(40, 100, n).astype(float),
    'Science_Score': np.random.randint(40, 100, n).astype(float),
    'Attendance_%': np.random.uniform(50, 100, n)
}

df = pd.DataFrame(data)

# Introduce missing values
df.loc[np.random.choice(n, 10, replace=False), 'Math_Score'] = np.nan
df.loc[np.random.choice(n, 5, replace=False), 'Age'] = np.nan

# Introduce outliers
df.loc[5, 'Math_Score'] = 200   # impossible score
df.loc[10, 'Attendance_%'] = 150  # impossible attendance

print('Dataset Created!')
df.head(10)

## Step 1: Scan for Missing Values and Inconsistencies

We check missing values using `isnull().sum()` and fix the `Gender` inconsistency (M/Male, F/Female).

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

# Fix Gender inconsistency
df['Gender'] = df['Gender'].replace({'M': 'Male', 'F': 'Female'})
df['Gender'].fillna('Unknown', inplace=True)

# Fill missing numeric values with median
df['Math_Score'].fillna(df['Math_Score'].median(), inplace=True)
df['Age'].fillna(df['Age'].median(), inplace=True)

print('\nAfter fixing missing values:')
print(df.isnull().sum())
print('\nGender unique values:', df['Gender'].unique())

## Step 2: Detect and Handle Outliers

We use the **IQR (Interquartile Range)** method to detect outliers. Values beyond Q1-1.5*IQR or Q3+1.5*IQR are capped (Winsorization).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df.boxplot(column='Math_Score', ax=axes[0])
axes[0].set_title('Math Score (Before)')
df.boxplot(column='Attendance_%', ax=axes[1])
axes[1].set_title('Attendance % (Before)')
plt.tight_layout()
plt.show()

In [ ]:
def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series.clip(lower=lower, upper=upper)

df['Math_Score'] = cap_outliers(df['Math_Score'])
df['Attendance_%'] = cap_outliers(df['Attendance_%'])
print('Outliers capped using IQR method!')
print(df[['Math_Score', 'Attendance_%']].describe())

## Step 3: Data Transformation

**Reason:** The `Attendance_%` column may be slightly skewed. We apply a **log transformation** to reduce skewness and bring the distribution closer to normal. This is useful for ML models that assume normality.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df['Attendance_%'], bins=15, color='skyblue', edgecolor='black')
axes[0].set_title('Attendance % - Original')

# Log transformation (add 1 to avoid log(0))
df['Attendance_log'] = np.log1p(df['Attendance_%'])

axes[1].hist(df['Attendance_log'], bins=15, color='salmon', edgecolor='black')
axes[1].set_title('Attendance % - Log Transformed')

plt.tight_layout()
plt.show()

print(f'Original Skewness: {df["Attendance_%"].skew():.3f}')
print(f'Log Transformed Skewness: {df["Attendance_log"].skew():.3f}')